### Agglomerative Clustering from scratch

In [2]:
import numpy as np

class AgglomerativeClusteringScratch:
  def __init__(self, n_clusters=None, linkage='average'):
    # Use euclidean matric for computing pairwise distance matrix
    self.n_clusters = n_clusters

    if linkage == 'complete':
      self.linkage_distance_func = self.max_linkage_distance
    elif linkage == 'single':
      self.linkage_distance_func = self.min_linkage_distance
    elif linkage == 'average':
      self.linkage_distance_func = self.avg_linkage_distance
    elif linkage == 'ward':
      self.linkage_distance_func = self.ward_method_distance

  def avg_linkage_distance(self, cluster_A, cluster_B):
    # Compute average linkage distance between two clusters
    distance = 0
    for i in range(cluster_A.shape[0]):
      distance += np.linalg.norm(cluster_B - cluster_A[i, :], axis=1).sum()
    distance /= (cluster_A.shape[0] * cluster_B.shape[0])
    return distance

  def max_linkage_distance(self, cluster_A, cluster_B):
    # Compute maximum linkage distance between two clusters
    distance = 0
    for i in range(cluster_A.shape[0]):
      distance = np.append(np.linalg.norm(cluster_B - cluster_A[i, :], axis=1), distance).max()
    return distance

  def min_linkage_distance(self, cluster_A, cluster_B):
    # Compute minimum linkage distance between two clusters
    distance = np.inf
    for i in range(cluster_A.shape[0]):
      distance = np.append(np.linalg.norm(cluster_B - cluster_A[i, :], axis=1), distance).min()
    return distance

  def ward_method_distance(self, cluster_A, cluster_B):
    # Compute the Ward linkage distance between two clusters
    n_A = cluster_A.shape[0]
    n_B = cluster_B.shape[0]

    centroid_A = np.mean(cluster_A, axis=0)
    centroid_B = np.mean(cluster_B, axis=0)

    # Distance is proportional to the squared Euclidean distance between centroids
    # scaled by size of the clusters
    diff = centroid_A - centroid_B
    distance = (n_A * n_B) / (n_A + n_B) * np.dot(diff, diff)

    return distance

  def pairwise_distance(self, data, n_samples):
    # Compute the pairwise distance matrix in euclidean matric
    distance_mat = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
      for j in range(i + 1, n_samples):
        distance = np.linalg.norm(data[i] - data[j])
        distance_mat[i, j] = distance
        distance_mat[j, i] = distance
    return distance_mat

  def update(self, data, distance_mat, labels):
      #"Find closest clusters, merge clusters, delete cluster, update distance"
      idx_upper = np.triu_indices(distance_mat.shape[0], k=1)  # Index of upper part of distance matrix (skip diagonal)
      min_value = np.min(distance_mat[idx_upper])  # Value of idx_upper
      row, col = np.argwhere(distance_mat == min_value)[0]  # Index of min_value (same as d_kl)

      # Update label
      labels[labels == col] = row
      labels[labels > col] -= 1

      # Deleted the row and column 'col'
      distance_mat = np.delete(distance_mat, col, 0)
      distance_mat = np.delete(distance_mat, col, 1)

      # Update distance matrix
      for i in range(len(distance_mat)):
          distance_mat[row, i] = self.linkage_distance_func(data[labels == row], data[labels == i])
          distance_mat[i, row] = distance_mat[row, i]
      return distance_mat, labels

  def fit_predict(self, X):
    self.data = X
    self.n_samples = self.data.shape[0]
    self.initial_distance = self.pairwise_distance(self.data, self.n_samples)
    self.labels = np.arange(self.n_samples)
    self.distance_matrix = self.initial_distance.copy()
    while len(np.unique(self.labels)) > self.n_clusters:
      # Fill in the diagonal as infinity to determine that the distance is the same position.
      np.fill_diagonal(self.distance_matrix, np.inf)
      self.distance_matrix, self.labels = self.update(self.data, self.distance_matrix, self.labels)

    return self.labels

# Attrition Prediction using Agglomerative Clustering
Load the SMOTE-balanced attrition dataset and apply agglomerative clustering.  
**Note:** Agglomerative clustering is $O(n^3)$, so we sample the dataset to keep runtime reasonable.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the SMOTE-balanced attrition dataset
df = pd.read_csv('../data/processed_data_attrition.csv')

# Separate features and Attrition label
attrition_raw = df['Attrition'].values
attrition_binary = (attrition_raw == attrition_raw.max()).astype(int)  # 1 = Yes, 0 = No
X_full = df.drop(columns=['Attrition']).values
feature_names = df.drop(columns=['Attrition']).columns.tolist()

print(f"Full dataset shape: {X_full.shape}")
print(f"Attrition distribution: No={np.sum(attrition_binary == 0)}, Yes={np.sum(attrition_binary == 1)}")

# Sample the dataset for agglomerative clustering (O(n^3) is too slow on full data)
SAMPLE_SIZE = 500
np.random.seed(42)
sample_idx = np.random.choice(len(X_full), SAMPLE_SIZE, replace=False)
X = X_full[sample_idx]
y_true = attrition_binary[sample_idx]

print(f"\nSampled {SAMPLE_SIZE} points for clustering")
print(f"Sample attrition distribution: No={np.sum(y_true == 0)}, Yes={np.sum(y_true == 1)}")

Full dataset shape: (2464, 36)
Attrition distribution: No=1232, Yes=1232

Sampled 500 points for clustering
Sample attrition distribution: No=255, Yes=245


In [4]:
# Run Agglomerative Clustering (K=2 for Attrition: Yes/No)
print("Running Agglomerative Clustering (ward linkage, K=2)...")
print("This may take a few minutes on 500 samples...")

agg = AgglomerativeClusteringScratch(n_clusters=2, linkage='ward')
labels = agg.fit_predict(X)

print(f"\nClustering complete!")
print(f"Cluster distribution:")
for k in np.unique(labels):
    count = np.sum(labels == k)
    print(f"  Cluster {k}: {count} samples ({count/len(labels)*100:.1f}%)")

Running Agglomerative Clustering (ward linkage, K=2)...
This may take a few minutes on 500 samples...

Clustering complete!
Cluster distribution:
  Cluster 0: 209 samples (41.8%)
  Cluster 1: 291 samples (58.2%)


In [5]:
# Evaluate: map clusters to attrition labels and compute metrics
# Since clustering is unsupervised, we need to find the best mapping
# between cluster IDs and actual attrition labels

def map_clusters_to_labels(cluster_labels, true_labels):
    """Find the cluster-to-label mapping that maximizes accuracy."""
    unique_clusters = np.unique(cluster_labels)
    # Try both mappings (cluster 0->0, 1->1) and (cluster 0->1, 1->0)
    mapping_1 = cluster_labels.copy()
    acc_1 = np.mean(mapping_1 == true_labels)

    mapping_2 = 1 - cluster_labels  # flip
    acc_2 = np.mean(mapping_2 == true_labels)

    if acc_2 > acc_1:
        return mapping_2, acc_2
    return mapping_1, acc_1

predicted, accuracy = map_clusters_to_labels(labels, y_true)

# Confusion Matrix
tp = np.sum((predicted == 1) & (y_true == 1))
tn = np.sum((predicted == 0) & (y_true == 0))
fp = np.sum((predicted == 1) & (y_true == 0))
fn = np.sum((predicted == 0) & (y_true == 1))

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("=" * 50)
print("  Agglomerative Clustering — Attrition Evaluation")
print("=" * 50)
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"\n  Confusion Matrix:")
print(f"                Predicted")
print(f"                No(0)  Yes(1)")
print(f"  Actual No(0)  {tn:<6} {fp}")
print(f"  Actual Yes(1) {fn:<6} {tp}")

# Attrition rate per cluster
print(f"\n  Attrition Rate per Cluster (before mapping):")
for k in np.unique(labels):
    cluster_mask = labels == k
    attrition_rate = np.mean(y_true[cluster_mask]) * 100
    print(f"    Cluster {k}: {attrition_rate:.1f}% attrition "
          f"({np.sum(y_true[cluster_mask])}/{np.sum(cluster_mask)} employees)")

  Agglomerative Clustering — Attrition Evaluation
  Accuracy:  0.5840
  Precision: 0.5636
  Recall:    0.6694
  F1-Score:  0.6119

  Confusion Matrix:
                Predicted
                No(0)  Yes(1)
  Actual No(0)  128    127
  Actual Yes(1) 81     164

  Attrition Rate per Cluster (before mapping):
    Cluster 0: 38.8% attrition (81/209 employees)
    Cluster 1: 56.4% attrition (164/291 employees)


In [6]:
# Try different linkage methods and compare
print("Comparing Linkage Methods (K=2):\n")
print(f"{'Linkage':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 60)

for linkage in ['single', 'complete', 'average', 'ward']:
    agg_temp = AgglomerativeClusteringScratch(n_clusters=2, linkage=linkage)
    temp_labels = agg_temp.fit_predict(X)
    temp_pred, temp_acc = map_clusters_to_labels(temp_labels, y_true)

    tp_t = np.sum((temp_pred == 1) & (y_true == 1))
    fp_t = np.sum((temp_pred == 1) & (y_true == 0))
    fn_t = np.sum((temp_pred == 0) & (y_true == 1))
    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    f1_t = 2 * prec_t * rec_t / (prec_t + rec_t) if (prec_t + rec_t) > 0 else 0

    print(f"{linkage:<12} {temp_acc:<12.4f} {prec_t:<12.4f} {rec_t:<12.4f} {f1_t:<12.4f}")

Comparing Linkage Methods (K=2):

Linkage      Accuracy     Precision    Recall       F1-Score    
------------------------------------------------------------
single       0.5080       0.0000       0.0000       0.0000      
complete     0.5280       0.5096       0.9755       0.6695      
average      0.5060       0.3750       0.0122       0.0237      
ward         0.5840       0.5636       0.6694       0.6119      


In [7]:
# Compare with sklearn AgglomerativeClustering as baseline
from sklearn.cluster import AgglomerativeClustering

sklearn_agg = AgglomerativeClustering(n_clusters=2, linkage='ward')
sklearn_labels = sklearn_agg.fit_predict(X)
sklearn_pred, sklearn_acc = map_clusters_to_labels(sklearn_labels, y_true)

tp_sk = np.sum((sklearn_pred == 1) & (y_true == 1))
tn_sk = np.sum((sklearn_pred == 0) & (y_true == 0))
fp_sk = np.sum((sklearn_pred == 1) & (y_true == 0))
fn_sk = np.sum((sklearn_pred == 0) & (y_true == 1))
prec_sk = tp_sk / (tp_sk + fp_sk) if (tp_sk + fp_sk) > 0 else 0
rec_sk = tp_sk / (tp_sk + fn_sk) if (tp_sk + fn_sk) > 0 else 0
f1_sk = 2 * prec_sk * rec_sk / (prec_sk + rec_sk) if (prec_sk + rec_sk) > 0 else 0

print("=" * 50)
print("  Comparison: From-Scratch vs Sklearn (Ward, K=2)")
print("=" * 50)
print(f"  {'Metric':<12} {'From-Scratch':<15} {'Sklearn':<15}")
print(f"  {'-'*42}")
print(f"  {'Accuracy':<12} {accuracy:<15.4f} {sklearn_acc:<15.4f}")
print(f"  {'Precision':<12} {precision:<15.4f} {prec_sk:<15.4f}")
print(f"  {'Recall':<12} {recall:<15.4f} {rec_sk:<15.4f}")
print(f"  {'F1-Score':<12} {f1:<15.4f} {f1_sk:<15.4f}")

  Comparison: From-Scratch vs Sklearn (Ward, K=2)
  Metric       From-Scratch    Sklearn        
  ------------------------------------------
  Accuracy     0.5840          0.5840         
  Precision    0.5636          0.5837         
  Recall       0.6694          0.5265         
  F1-Score     0.6119          0.5536         
